In [ ]:





# ------------------------------------------
# 1. Use validation data only
# ------------------------------------------

baseline = models["Baseline"]

baseline_probs = baseline.predict_proba(X_val)[:, 1]

thresholds = np.round(
    np.arange(0.05, 0.951, 0.01), 2
)

# ------------------------------------------
# 2. Evaluate different thresholds
# ------------------------------------------

def calculate_metrics(y_true, probs, threshold):

    predictions = (probs >= threshold).astype(int)

    return {
        "Accuracy": accuracy_score(y_true, predictions),
        "Precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_true, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_true, predictions, zero_division=0
        )
    }


baseline_rows = []

for threshold in thresholds:

    scores = calculate_metrics(
        y_val, baseline_probs, threshold
    )

    baseline_rows.append({
        "Threshold": threshold,
        **scores
    })

baseline_results = pd.DataFrame(baseline_rows)

# ------------------------------------------
# 3. Find best threshold using validation F1
# ------------------------------------------

best_idx = baseline_results["F1"].idxmax()

best_baseline = baseline_results.loc[best_idx]

print("Best Baseline Threshold:",
      best_baseline["Threshold"])

print("Best Validation F1:",
      round(best_baseline["F1"], 3))

print("\nSelected Threshold Results:")

display(
    baseline_results[
        baseline_results["Threshold"].isin(
            [0.1, 0.2, 0.3, 0.4, 0.5]
        )
    ].round(3)
)

# ------------------------------------------
# 4. Plot threshold impact
# ------------------------------------------

plt.figure(figsize=(10, 6))

plt.plot(
    baseline_results["Threshold"],
    baseline_results["Precision"],
    label="Precision"
)

plt.plot(
    baseline_results["Threshold"],
    baseline_results["Recall"],
    label="Recall"
)

plt.plot(
    baseline_results["Threshold"],
    baseline_results["F1"],
    label="F1-score",
    linewidth=2
)

plt.axvline(
    x=best_baseline["Threshold"],
    linestyle="--",
    label=f"Best Threshold = {best_baseline['Threshold']:.2f}"
)

plt.xlabel("Decision Threshold")
plt.ylabel("Score")
plt.title("Independent Threshold Analysis - Baseline")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


NameError: name 'models' is not defined

In [ ]:

# ------------------------------------------
# 1. Get validation probabilities
# ------------------------------------------

validation_probs = {
    name: model.predict_proba(X_val)[:, 1]
    for name, model in models.items()
}

# ------------------------------------------
# 2. Evaluate all methods and thresholds
# ------------------------------------------

all_rows = []

for name, probs in validation_probs.items():

    for threshold in thresholds:

        scores = calculate_metrics(
            y_val, probs, threshold
        )

        all_rows.append({
            "Method": name,
            "Threshold": threshold,
            **scores
        })

comparison_df = pd.DataFrame(all_rows)

# ------------------------------------------
# 3. Compare methods at fixed threshold
# ------------------------------------------

fixed_results = comparison_df[
    np.isclose(comparison_df["Threshold"], 0.5)
].copy()

print("Fixed Threshold Comparison (0.5):")

display(
    fixed_results.sort_values(
        "F1", ascending=False
    ).round(3)
)

# ------------------------------------------
# 4. Find best threshold for each method
# ------------------------------------------

best_indices = comparison_df.groupby(
    "Method"
)["F1"].idxmax()

best_results = comparison_df.loc[
    best_indices
].copy()

best_results = best_results.sort_values(
    "F1", ascending=False
)

print("\nOptimized Threshold Comparison:")

display(best_results.round(3))

# ------------------------------------------
# 5. Plot F1 vs Threshold for all methods
# ------------------------------------------

plt.figure(figsize=(10, 6))

for name in models.keys():

    data = comparison_df[
        comparison_df["Method"] == name
    ]

    plt.plot(
        data["Threshold"],
        data["F1"],
        label=name,
        linewidth=2
    )

plt.axvline(
    0.5,
    linestyle="--",
    color="gray",
    label="Default Threshold"
)

plt.xlabel("Decision Threshold")
plt.ylabel("F1-score")
plt.title("Threshold Optimization vs Resampling")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:

# ------------------------------------------
# Select best model + threshold using F1
# ------------------------------------------

selected = best_results.iloc[0]

best_method = selected["Method"]
best_threshold = float(selected["Threshold"])

best_model = models[best_method]

print("Selected Method:", best_method)
print("Selected Threshold:", best_threshold)

# ------------------------------------------
# Independent test evaluation
# ------------------------------------------

test_probs = best_model.predict_proba(X_test)[:, 1]

test_scores = calculate_metrics(
    y_test, test_probs, best_threshold
)

test_scores["AP"] = average_precision_score(
    y_test, test_probs
)

test_scores["ROC-AUC"] = roc_auc_score(
    y_test, test_probs
)

print("\nFinal Test Results:")
display(pd.Series(test_scores).round(3))

test_predictions = (
    test_probs >= best_threshold
).astype(int)

print("\nFinal Confusion Matrix:")
print(confusion_matrix(y_test, test_predictions))
